## MSE 590400 coding assignment 


# Monte Carlo: Diffusion limited aggregation


#### <p style="text-align: right;"> &#9989; **put your name here** </p>



<div align="left">
<img src="https://upload.wikimedia.org/wikipedia/commons/b/b8/DLA_Cluster.JPG" width="400">
</div>
image from wikipedia

---


Diffusion limited aggregation (DLA) is an interesting materials growth phenomena. As indicated by the name, it is a diffusion dominated process; i.e., attaching is a much fast process than diffusion in this case. The diffusers are very easy to attach to the aggregation and stop transport. 

In this Monte Carlo simulation. We'll combine random walk model and sampling model (in some sophisticated ones). This assignment is built following 
<a href="https://github.com/huichiayu/cmse_202_802/blob/main/MSE590/A_Study_in_Monte_Carlo_Simulation_of_Modified_DLA.pdf"
   target="_blank" rel="noopener">this paper.</a>



The implementation of a simple DLA MC model can be decomposed to the following steps.

1.	Create a nucleus in the computational domain.

2.	Insert walkers into the system.

3.	The walkers randomly move within the domain. In some cases, the walking can be biased assuming it is affected by external field.

4. Impose boundary conditions.

5.	Examine the neighboring sites of each walker. If it satisfies the aggregation criteria, attach the walker and stop its future walk.

7.	Repeat procedure 2 through 4.

### Part 1. 

* Use the code cell below to initiate a 2D grid system. If a grid point is occupied by aggregation, put a value of `1`. Otherwise, set the value to `0`. Let's place a small cluster of 2x2 grid spacings in the middle of the 2D grid.

* Create a container array to store the information of walkers. Each walker has 3 values: The 1st one is $x$ position, the 2nd is $y$ position, and the 3rd is whether it's active. We'll use `1` for active and `0` for inactive.

* Visualize the walkers on the grid.

In [ ]:
import numpy as np
import random
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
import time


# fill ?? to complete the code

# domain size
Lx = 100
Ly = 100

# create the 2D domain
agr = np.zeros((??, ??))

# if the position is occupied by aggregation, the value will be 1.
# place an initial cluster in the middle of the domain
agr[??, ??] = 1


# initial number of walkers
N_init = 1000

# array to store walker information
wk = np.zeros((N_init,3),dtype=int)

# randomly assign initial position
# all walkers are active initially
for w in range(wk.shape[0]):
    wk[w,0] = ??
    wk[w,1] = ??
    wk[w,2] = 1




# visualize walkers and aggregation
fig, ax = plt.subplots(figsize=(4, 4))
ax.set_aspect('equal', adjustable='box')
ax.set_xlim(0, Lx)
ax.set_ylim(0, Ly)

sc = ax.scatter(wk[:, 0], wk[:, 1], s=8)
im = ax.imshow(agr, origin='lower', interpolation='nearest')  

title = ax.set_title("Step 0")

---
### Part 2

The code cell below is a function to supply walkers at random positions. Since walkers will attach to the aggregation and become inactive, we continue to supply walkers to the system. New walkers are injected to the system in the region $R_{min} < r < R_{max}$, where $r$ is the radius from the center of the domain. You may recognize that the function randomly select an angle and randomly select a radius between $R_{min}$ and $R_{max}$ for the position of a new walker. 

In [ ]:
# this function is used to supply walkers at a random position

Rmax = 50
Rmin = 15

def sample_in_annulus(Rmin, Rmax, cx=50.0, cy=50.0, rng=None):
    rng = np.random.default_rng(rng)
    theta = rng.uniform(0, 2*np.pi)
    r = np.sqrt(rng.uniform(Rmin*Rmin, Rmax*Rmax))   # uniform over area
    return np.array([cx + int(r*np.cos(theta)), cy + int(r*np.sin(theta))])


**Monte Carlo step**

Similar to the 2D random walk example code in previous class. We first create 4 sets of jumps on 2D. Then, the random walk step will pick one of the four sets to jump. 

The following code is divided into 6 sections.
1. Section 1: We need to set up a stop criterion for the simulation. Here, we use a commonly used Python technique -- mask, which returns a bool indicating whether the imposed the condition is satisfied. The code in section 1 does: For `agr`, return `true` or `false` for the column containing nonzeros. `axis = 0` means a column. Then, find the indices of columns that has `true`: the first one will the westmost column and the last one will be the eastmost column. We stop the simulation if the aggregation reaches the domain box. Thus, when the westmost column reaches 1 or the eastmost column reach Lx-2. Stop the `for` loop using `break`.
5. Section 2: We create a mask to check wether a walker is active. If active, it returns `true`. Otherwise `false`. Then, we keep only active walkers. Next, loop over active walkers and make a random jump.
6. Section 3: Impose periodic boundary condtions. If a walker leaves the box from a boundary, it comes back from the opposite boundary.
7. Section 4: Check whether a walker is adjacent to the aggregation. For example, if a walker's west side is the aggregation, the walker attaches; the walker becomes inactive and the aggregation grows in the expense of that walker.
8. Section 5: Every once in a while, Check how many walkers are dead. Create an array for new walkers. Calculate the range to supply new walkers. Check whether the new walkers are on the boundaries. Finally, combine the new walker array to the existing walker array using `vstack`.
9. Section 6: visualization.

In [ ]:
nstep = 2001

# random jump in 2D 
dirs = np.array([[+1, 0], [-1, 0], [0, +1], [0, -1]], dtype=int)

# Monte Carlo step
for it in range(1, nstep):

    # section 1 -------------------------------------------------
    # stopping criteria
    mask = agr.any(axis=0)  
    cols = np.flatnonzero(mask)         # indices of such columns
    if cols[0] == 1 or cols[-1] == Lx-2:
        ??  # <-- stop the loop
    
    mask = agr.any(axis=1)
    rows = np.flatnonzero(mask)         # indices of such row
    if rows[0] == 1 or rows[-1] == Ly-2:
        ??

    # section 2 -------------------------------------------------
    # create a mask to examine whether a walker is active
    mask = wk[:,2] == 1
    # the items satisfy the mask will be labeled true.
    # i.e., keep only active ones
    wk = wk[mask]

    # number of active walker
    N = wk.shape[0]
    # loop over all active walkers
    for w in range(N):
        
        # pick a random direction for each walker and step by 1
        step_idx = ??       
        wk[w,0:2] += dirs[step_idx]

        # section 3 --------------------------------------------
        # impose boundary conditions
        if wk[w,0] == 0:     # reach west boundary
            wk[w,0] = ??
        if wk[w,0] == Lx-1:  # reach east boundary
            wk[w,0] = ??

        if wk[w,1] == 0:     # reach south boundary
            wk[w,1] = ??
        if wk[w,1] == Ly-1:  # reach north boundary
            wk[w,1] = ??        

        # section 4 -------------------------------------------
        # the walker position at this jump
        rw = wk[w,0] 
        cl = wk[w,1] 
      
        # examine whether the walker jump to a site next to the aggregation
        # check west of the walker
        if agr[??,??] == 1:
            wk[w,2] = ?? 
            agr[rw,cl] = ??

        # check east of the walker
        ??

        # check south 
        ??
                    
        # check north
        ??
            
    # section 5 -------------------------------------------
    # supply new walkers
    if it%100 == 0:
        # find how many new walkers needed
        nW = N_init - N

        # create a new walker array
        n_wk = np.zeros((??,??), dtype=int)

        # define the minimum radius away from the domain center
        Rmin = np.sqrt(np.sum(agr)/np.pi)*1.2
        
        for k in range(nW):
            
            xi, yi = sample_in_annulus(Rmin, Rmax, cx=50.0, cy=50.0, rng=None)
            
            # check boundary conditions
            if xi == 0:
                xi = ??
            if xi == Lx-1:
                xi = ??

            if yi == 0:
                yi = ??
            if yi == Ly-1:
                yi = ??
            # define the position of new walkers
            n_wk[k,0] = xi
            n_wk[k,1] = yi
            n_wk[k,2] = ??        

        # combine the new walker array to the old one
        wk = np.vstack([wk, n_wk],dtype=int)     

    # section 6 --------------------------------------------------------
    if it%5==0:
        # visualization
        sc.set_offsets(wk)  
        im.set_data(agr)
        title.set_text(f"Step {it}" )
        
        # Animaiton part (dosn't change)
        clear_output(wait=True) # Clear output for dynamic display
        display(fig)            # Reset display
        # fig.clear()             # Prevent overlapping and layered plots
        time.sleep(0.0002)         # Sleep for half a second to slow down the animation

---
### Part 3

Add a new feature for reaction-limited aggregation. Now you need to count how many times a site is visited. This array `nsv` can be in the same form of `agr`. Starting with all zeros on `nsv`. 

* If a site is visited, you add 1 on that position `nsv[rw,cl]`. This check can be implemented when a walker reaches a grid point next to the aggregation.
* Walker attachment will occur only when a site has been visited more than `thres` times. 

**A large value of `thres` mimics the process of near equilibrium condition.** In this case, attachement becomes the limiting factor of growth.

Copy your functioning code above to the code cell below and include the new feature.

In [ ]:



# create the 2D domain
nsv = np.zeros((Ly,Lx))

thres = 10

nstep = 10001



# random jump in 2D 
dirs = np.array([[+1, 0], [-1, 0], [0, +1], [0, -1]], dtype=int)

# Monte Carlo step
for it in range(1, nstep):
    # copy your functioning code here and made necessary modifications 
    
        

&#9989; Do This - Describe what you observed in this Markdown cell. Make a comparison between the morphologies from diffusion-limitied and reaction-limited cases. 

---
### Great! You're done! Please add your name to the file name and upload your file to the respective drop box. This assignment is due on 10/6. 